In [24]:

#This version is for the case where we SKIP flipping the input  

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, solve_ivp
%matplotlib inline
from pathlib import Path




from scipy.integrate import quad
from scipy.optimize import minimize_scalar

def optimize_tau_continuous(coutfuncFlip, h, T, Tf, tau_min, tau_max, epsabs=1e-12):
    """
    Maximize
        I(tau) = \int_0^Tf coutfuncFlip[-2](tt + tau) * h(tt, T) dtt
    for tau in [tau_min, tau_max].

    Returns
    -------
    tau_opt : float
        Value of tau that maximizes the integral
    val_opt : float
        Maximum integral value
    err_opt : float
        quad error estimate at tau_opt
    result : OptimizeResult
        Full scipy optimization result
    """
    f = coutfuncFlip[-2]

    def objective(tau):
        val, _ = quad(
            lambda tt: f(tt ) * h(tt+ tau, T),
            0, Tf,
            epsabs=epsabs
        )
        return -val

    result = minimize_scalar(
        objective,
        bounds=(tau_min, tau_max),
        method="bounded"
    )

    tau_opt = result.x
    val_opt, err_opt = quad(
        lambda tt: f(tt ) * h(tt+ tau_opt, T),
        0, Tf,
        epsabs=epsabs
    )

    return tau_opt, val_opt, err_opt, result



def argmax_output_time(tt_grid, coutfuncFlip):
    f = lambda tt: np.abs(coutfuncFlip[-2](tt))
    vals = np.array([f(tt) for tt in tt_grid])
    return tt_grid[np.argmax(vals)]


def h(t, T):
    """
    Smooth, time-symmetric, normalized pulse:
        h(t,T) = sqrt(8/(3T)) * sin(pi*t/T)^2
    """
    return (np.sqrt(8/(3*T)) * np.sin(np.pi * t / T)**2)*np.heaviside(t, 0.0) * np.heaviside(T - t, 0.0)


# def Cop(k, g, gam):
#     return (g*g)/(k*gam)

# def etaM(ki, kx, g, gam):
#     return (kx/(ki+kx))*(Cop(ki+ kx, g, gam)/(1+Cop(ki+ kx, g, gam)))


# def Omega1(t, g, T, chi, eta, kx):
#     return -np.exp(1j * chi)*np.sqrt(eta)*g*h(t, T)/np.sqrt(2 * kx)

# def top(t, g, T, chi, eta, kx):
#     return -np.exp(1j*chi)*np.sqrt(eta/(2*kx))*h(t, T)*np.sqrt((g**2+(ki+kx)*gam)**2)*(1/g)

# def bottom2(t, g, T, chi, eta, kx):
#     return (1-(eta/etaM(ki, kx, g, gam))*(12*np.pi*t-8*T*np.sin(2*np.pi*t/T)+T*np.sin(4*np.pi*t/T))/(12*np.pi*T))

def expr1( t, g1, T1, eta1, kx1, gam1, ki1):
    return -(1.0 / (12 * g1**2 * kx1 * np.pi * T1**3)) * (
        8 * eta1 * np.pi**3
        + 16 * eta1 * gam1 * np.pi**3 * t
        + 6 * eta1 * g1**2 * np.pi * T1**2
        + 12 * eta1 * gam1 * ki1 * np.pi * T1**2
        + 6 * eta1 * ki1**2 * np.pi * T1**2
        + 12 * eta1 * gam1 * kx1 * np.pi * T1**2
        + 12 * eta1 * ki1 * kx1 * np.pi * T1**2
        + 6 * eta1 * kx1**2 * np.pi * T1**2
        + 12 * eta1 * g1**2 * ki1 * np.pi * t * T1**2
        + 12 * eta1 * gam1 * ki1**2 * np.pi * t * T1**2
        + 12 * eta1 * g1**2 * kx1 * np.pi * t * T1**2
        + 24 * eta1 * gam1 * ki1 * kx1 * np.pi * t * T1**2
        + 12 * eta1 * gam1 * kx1**2 * np.pi * t * T1**2
        - 12 * g1**2 * kx1 * np.pi * T1**3
        - 8 * eta1 * (g1**2 + (ki1 + kx1) * (2 * gam1 + ki1 + kx1))
          * np.pi * T1**2 * np.cos((2 * np.pi * t) / T1)
        - 2 * eta1 * np.pi * (
            4 * np.pi**2
            - (g1**2 + (ki1 + kx1) * (2 * gam1 + ki1 + kx1)) * T1**2
        ) * np.cos((4 * np.pi * t) / T1)
        + 16 * eta1 * ki1 * np.pi**2 * T1 * np.sin((2 * np.pi * t) / T1)
        + 16 * eta1 * kx1 * np.pi**2 * T1 * np.sin((2 * np.pi * t) / T1)
        - 8 * eta1 * g1**2 * ki1 * T1**3 * np.sin((2 * np.pi * t) / T1)
        - 8 * eta1 * gam1 * ki1**2 * T1**3 * np.sin((2 * np.pi * t) / T1)
        - 8 * eta1 * g1**2 * kx1 * T1**3 * np.sin((2 * np.pi * t) / T1)
        - 16 * eta1 * gam1 * ki1 * kx1 * T1**3 * np.sin((2 * np.pi * t) / T1)
        - 8 * eta1 * gam1 * kx1**2 * T1**3 * np.sin((2 * np.pi * t) / T1)
        - 4 * eta1 * gam1 * np.pi**2 * T1 * np.sin((4 * np.pi * t) / T1)
        - 8 * eta1 * ki1 * np.pi**2 * T1 * np.sin((4 * np.pi * t) / T1)
        - 8 * eta1 * kx1 * np.pi**2 * T1 * np.sin((4 * np.pi * t) / T1)
        + eta1 * g1**2 * ki1 * T1**3 * np.sin((4 * np.pi * t) / T1)
        + eta1 * gam1 * ki1**2 * T1**3 * np.sin((4 * np.pi * t) / T1)
        + eta1 * g1**2 * kx1 * T1**3 * np.sin((4 * np.pi * t) / T1)
        + 2 * eta1 * gam1 * ki1 * kx1 * T1**3 * np.sin((4 * np.pi * t) / T1)
        + eta1 * gam1 * kx1**2 * T1**3 * np.sin((4 * np.pi * t) / T1)
    )

def expr2(t, g1, T1, eta1, kx1, gam1, ki1):
    num = (
        np.sqrt(eta1 / kx1)
        * (1.0 / T1) ** 2.5
        * (
            (-4 * np.pi**2 + (g1**2 + gam1 * (ki1 + kx1)) * T1**2)
            * np.cos((2 * np.pi * t) / T1)
            - T1
            * (
                (g1**2 + gam1 * (ki1 + kx1)) * T1
                + 2 * (gam1 + ki1 + kx1) * np.pi * np.sin((2 * np.pi * t) / T1)
            )
        )
    )
    den = np.sqrt(3.0) * g1
    return num / den

def Omega1(t, g1, T1, chi, eta1, kx1, gam1, ki1):
    return -np.abs(expr2(t, g1, T1, eta1, kx1, gam1, ki1))/np.sqrt(expr1(t, g1, T1, eta1, kx1, gam1, ki1))*np.heaviside(t, 0.0) * np.heaviside(T1 - t, 0.0)


def rhs(t, y, g, ki, kx, gam, chi, eta, ain, T):
    cs, ce, cg = y
    om = Omega1(t, g, T, chi, eta, kx, gam, ki)  # comment this out to see which one fails
    a_in_t = ain(t)
    return [-1j*om*ce, -1j*om*cs-1j*g*cg -gam*ce, 
            -1j*g*ce -(ki+kx)*cg-np.sqrt(2*kx)*a_in_t]

def rhs2(t, y,g, ki, kx, gam, chi, eta, ain, T):
    cs, ce, cg = y
    om = Omega1(T-t, g, T, chi, eta, kx, gam, ki)  # comment this out to see which one fails
    a_in_t = ain(t)
    return [-1j*om*ce, -1j*om*cs-1j*g*cg -gam*ce, 
            -1j*g*ce -(ki+kx)*cg-np.sqrt(2*kx)*a_in_t]

def SolA( g, ki, kx, gam, chi, eta, ain, init0, T, Tf):
    args1 = [g, ki, kx, gam, chi, eta, ain, T ]
    return solve_ivp(rhs, (0, Tf), init0, args=args1, dense_output=True,rtol=1e-12,   # default is 1e-3
    atol=1e-12)  # default is 1e-6)

def SolB( g, ki, kx, gam, chi, eta, ain, init0, T, Tf):
    # print("SolB: type(ain) =", type(ain))
    args1 = [g, ki, kx, gam, chi, eta, ain, T ]
    return solve_ivp(rhs2, (0, Tf), init0, args=args1, dense_output=True, rtol=1e-12,   # default is 1e-3
    atol=1e-12)  # default is 1e-6)



import matplotlib.pyplot as plt





def csPop(g, kx, ki, gam, T, kF, indexf):

    output_dir0 = Path(f"ineff_g_{g}")
    output_dir0.mkdir(exist_ok=True)
    
    output_dir = Path(f"plots_g_{g}_ind_{indexf}")
    output_dir.mkdir(exist_ok=True)

    output_dir2 = Path(f"topt_g_{g}")
    output_dir2.mkdir(exist_ok=True)

    chi=0
    coutAbsSq=[]
    csNoFlip=[]

    Tf=1.31*T
    t1 = np.linspace(0, Tf, 100)

    csFinalOccupation=[]

    for k in range(1, kF, 1):
        # print("k = ", k)
        eta=np.sin(np.pi/(2*(2*k+1)))**2

        cs0=1+0j

        ainfunc=[]
        csfunc=[]
        cefunc=[]
        cgfunc=[]
        coutfuncFlip=[]
        topt = []
        
        
        
        

        h1 = lambda t: 0
        t1 = np.linspace(0, Tf, 100)
        ainfunc.append(h1)

        for i in range(k+1):

            
            
            init0=np.array([cs0, 0+0j, 0+0j])
            solAA= SolA(g, ki, kx, gam, chi, eta, ainfunc[-1], init0, T, Tf)

            
            y = np.array([np.real(ainfunc[-1](tt)) for tt in t1])

            cs = lambda t, sol=solAA: sol.sol(t)[0] 
            csfunc.append(cs)
            ce = lambda t, sol=solAA: sol.sol(t)[1] 
            cefunc.append(ce)
            cg = lambda t, sol=solAA: sol.sol(t)[2] 
            cgfunc.append(cg)
            
            h1 = lambda t, f1=ainfunc[-1], g1=cgfunc[-1]: f1(t) + np.sqrt(2*kx)*g1(t)
            coutfuncFlip.append(h1)
            h2 = lambda t, f1=ainfunc[-1], g1=cgfunc[-1]: f1(t) + np.sqrt(2*kx)*g1(t)
            ainfunc.append(h2)
            coutfuncFlip.append(h1)
            
            if i==k:
            #     csFinalOccupation.append(csfunc[-1](T))
                # print("hello, adding", csfunc[-1](T))
                # print(csfunc[-1](Tf))
                # print("awdwad")
                csNoFlip.append(csfunc[-1](Tf))
                tau_opt, val_opt, err_opt, result = optimize_tau_continuous(
                coutfuncFlip, h, T, Tf,
                tau_min=-20,
                tau_max=20)
                #print("g, kx = ", (g, kx))
                                    
                #print(1-val_opt)
                coutAbsSq.append(1-val_opt)
                topt.append(tau_opt)





            
            # y = np.array([np.real(csfunc[-1](tt)) for tt in t1])
            # ya = np.array([np.real(coutfunc[-1](tt)) for tt in t1])
            # plt.figure()
            # plt.plot(t1, y)
            # # plt.plot(t1, ya)
            # plt.xlabel("t")
            # plt.ylabel("h1(t)")
            # plt.title("Plot of h1(t)")
            # plt.show()
            
            

            init0=np.array([csfunc[-1](Tf), 0+0j, 0+0j])
            solBB= SolB(g, ki, kx, gam, chi, eta, ainfunc[-1], init0, T, Tf)

            cs = lambda t, sol=solBB: sol.sol(t)[0] 
            csfunc.append(cs)
            ce = lambda t, sol=solBB: sol.sol(t)[1] 
            cefunc.append(ce)
            cg = lambda t, sol=solBB: sol.sol(t)[2] 
            cgfunc.append(cg)


            
            h1 = lambda t, f1=ainfunc[-1], g1=cgfunc[-1]: f1(t) + np.sqrt(2*kx)*g1(t)
            h2 = lambda t, f1=ainfunc[-1], g1=cgfunc[-1]: f1(t) + np.sqrt(2*kx)*g1(t)

            coutfuncFlip.append(h1)
            ainfunc.append(h2)

            
            cs0=csfunc[-1](Tf)
            

            # if i!=k:
            #     # print(cs0)
            #     csNoFlip.append(csfunc[-1](Tf))
            #     # y = np.array([np.real(csfunc[-1](tt)) for tt in t1])
            #     # ya = np.array([np.real(coutfunc[-1](tt)) for tt in t1])
            #     # plt.figure()
            #     # plt.plot(t1, y)
            #     # # plt.plot(t1, ya)
            #     # plt.xlabel("t")
            #     # plt.ylabel("h1(t)")
            #     # plt.title("Plot of h1(t)")
            #     # plt.show()
        #print("heyyy")
        #print(argmax_output_time(t1, coutfuncFlip))
        #tauu = T/2 - argmax_output_time(t1, coutfuncFlip)
        #print(tauu)     
        #integralSq, err = quad(lambda tt: coutfuncFlip[-2](tt)*h(tt+tauu, T), 0, Tf, epsabs=1e-12)

        
        # print(integralSq)
        
        # plt.figure()
        # plt.figure()
        # ya = np.array([np.real(coutfuncFlip[-2](tt)) for tt in t1]) #this is the second last. 
        # yb = np.array([np.real(h(tt , T)) for tt in t1]) #this is the second last. 
        # plt.plot(t1, y, color = 'black', linestyle = ':')
        # plt.plot(t1, ya,linestyle='--',color = 'green')
        # plt.xlabel("t")
        # plt.ylabel("h1(t)")
        # plt.show()
        # plt.tight_layout()
        # plt.show()
        


        plt.figure()
        #print("t1 min/max:", np.min(t1), np.max(t1))
        #print("tau_opt:", tau_opt)
        #print("shifted min/max:", np.min(t1 + tau_opt), np.max(t1 + tau_opt))
        
        ya = np.array([np.real(coutfuncFlip[-2](tt)) for tt in t1])
        yb = np.array([np.real(h(tt+tau_opt, T)) for tt in t1])
        yc = np.array([np.real(h(tt, T)) for tt in t1])

        plt.plot(t1, yb, linestyle='--', color='green')
        plt.plot(t1, ya, linestyle=':', color='red')
        plt.plot(t1, yc, linestyle=':', color='black')

        plt.xlabel("t")
        plt.ylabel("h1(t)")

        plt.tight_layout()
        plt.savefig(output_dir/f"plot_kx_{kx}_indF_{k}.png", dpi=200, bbox_inches="tight")
        plt.close()

    # print(coutAbsSq)
    # print(np.array(csNoFlip**2))



    




    #data = np.array(np.real(csNoFlip))**2
    data = coutAbsSq
    str1 = "g" +str(g)+ str(indexf)
    str2 = "g" + str(g)+ str(indexf)+"topt"
    print(str1)
    # Save as CSV
    np.savetxt(
        #"csFOkx1.csv",
        output_dir0 / f"{str1}.csv",
        data,
        delimiter=",",
        comments=""
    )

    np.savetxt(
        #"csFOkx1.csv",
        output_dir2 / f"{str2}.csv",
        topt,
        delimiter=",",
        comments=""
    )

 

 

<>:16: SyntaxWarning: invalid escape sequence '\i'
<>:16: SyntaxWarning: invalid escape sequence '\i'
/var/folders/g6/132z7dys28nd430vzzqgg8gh0000gn/T/ipykernel_39073/3799459384.py:16: SyntaxWarning: invalid escape sequence '\i'
  """


In [ ]:
g=6
ki=0
gam=0
T = 100
kF= 36


from joblib import Parallel, delayed

def run_case(kx, indF, g, ki, gam, T, kF):
    val = csPop(g, kx, ki, gam, T, kF, indF)
    return {"kx": kx, "indF": indF, "val": val}

cases = [(0.5 + 0.2*(i-1), i) for i in range(1,15)]

results = Parallel(n_jobs=-1)(
    delayed(run_case)(kx, indF, g, ki, gam, T, kF)
    for (kx, indF) in cases
)

results

/var/folders/g6/132z7dys28nd430vzzqgg8gh0000gn/T/ipykernel_39073/257254038.py:35: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
  the requested tolerance from being achieved.  The error may be 
  underestimated.


g61
g62
g63
g64
g65
g66
g67
g68
g610
g611
g69
g613
g612
g614


[{'kx': 0.5, 'indF': 1, 'val': None},
 {'kx': 0.7, 'indF': 2, 'val': None},
 {'kx': 0.9, 'indF': 3, 'val': None},
 {'kx': 1.1, 'indF': 4, 'val': None},
 {'kx': 1.3, 'indF': 5, 'val': None},
 {'kx': 1.5, 'indF': 6, 'val': None},
 {'kx': 1.7000000000000002, 'indF': 7, 'val': None},
 {'kx': 1.9000000000000001, 'indF': 8, 'val': None},
 {'kx': 2.1, 'indF': 9, 'val': None},
 {'kx': 2.3, 'indF': 10, 'val': None},
 {'kx': 2.5, 'indF': 11, 'val': None},
 {'kx': 2.7, 'indF': 12, 'val': None},
 {'kx': 2.9000000000000004, 'indF': 13, 'val': None},
 {'kx': 3.1, 'indF': 14, 'val': None}]

In [25]:
g=5
ki=0
gam=0
T = 100
kF=36


from joblib import Parallel, delayed

def run_case(kx, indF, g, ki, gam, T, kF):
    val = csPop(g, kx, ki, gam, T, kF, indF)
    return {"kx": kx, "indF": indF, "val": val}

cases = [(0.5 + 0.2*(i-1), i) for i in range(1,15)]

results = Parallel(n_jobs=-1)(
    delayed(run_case)(kx, indF, g, ki, gam, T, kF)
    for (kx, indF) in cases
)

results

/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/

KeyboardInterrupt: 

In [ ]:
g=4
ki=0
gam=0
T = 100
kF=36


from joblib import Parallel, delayed

def run_case(kx, indF, g, ki, gam, T, kF):
    val = csPop(g, kx, ki, gam, T, kF, indF)
    return {"kx": kx, "indF": indF, "val": val}

cases = [(0.5 + 0.2*(i-1), i) for i in range(1,15)]

results = Parallel(n_jobs=-1)(
    delayed(run_case)(kx, indF, g, ki, gam, T, kF)
    for (kx, indF) in cases
)

results

/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)
/Users/sharoona/Library/Python/3.12/lib/python/site-packages/scipy/integrate/_quadpack_py.py:577: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)


g41
g42
g43
g44
g45
g46
g47
g48
g411
g412
g410
g413
g49
g414


[{'kx': 0.5, 'indF': 1, 'val': None},
 {'kx': 0.7, 'indF': 2, 'val': None},
 {'kx': 0.9, 'indF': 3, 'val': None},
 {'kx': 1.1, 'indF': 4, 'val': None},
 {'kx': 1.3, 'indF': 5, 'val': None},
 {'kx': 1.5, 'indF': 6, 'val': None},
 {'kx': 1.7000000000000002, 'indF': 7, 'val': None},
 {'kx': 1.9000000000000001, 'indF': 8, 'val': None},
 {'kx': 2.1, 'indF': 9, 'val': None},
 {'kx': 2.3, 'indF': 10, 'val': None},
 {'kx': 2.5, 'indF': 11, 'val': None},
 {'kx': 2.7, 'indF': 12, 'val': None},
 {'kx': 2.9000000000000004, 'indF': 13, 'val': None},
 {'kx': 3.1, 'indF': 14, 'val': None}]

In [23]:
print("yoo")

yoo
